# 第三讲：NLP 基础与 Prompt 工程

**受众**：已完成第1-2讲学习的 Python 进阶学生
**前置条件**：熟悉 Python 基础语法、Pandas 基本操作；已配置 CloudStudio + CodeBuddy 环境
**学习目标**：

1. 理解语言模型演进与 Transformer 架构核心思想
2. 掌握分词原理（BPE 算法）并会用 tiktoken 计算 Token
3. 掌握采样参数（Temperature / Top-k / Top-p）的原理与调参
4. 掌握 Prompt Engineering 核心技巧（Zero/Few-shot、CoT、结构化输出）
5. 能通过 OpenAI 兼容 API 调用大模型完成三类 NLP 任务

**本讲两条主线**：

- 知识线：NLP 基础 → 分词 → 采样参数 → Prompt Engineering → API 调用 → 模型选择 → 幻觉
- 工具线：从 Vibe Coding 的两段式 Prompt 升级为完整的 Prompt Engineering 方法论

---

## Part 0：环境准备

**受众**：所有学生
**前置条件**：CloudStudio 工作区可用
**学习目标**：安装本讲所需依赖，配置 API 客户端


In [4]:
# 安装本讲依赖（首次运行执行一次即可）
# !pip install openai tiktoken -q

import json
import os

# 导入核心库
import tiktoken
from openai import OpenAI

print("[OK] 依赖导入成功")
print(
    f"tiktoken 版本: {tiktoken.__version__ if hasattr(tiktoken, '__version__') else '已安装'}")

[OK] 依赖导入成功
tiktoken 版本: 0.13.0


### 配置 API 客户端

课程统一使用 [硅基流动](https://cloud.siliconflow.cn/i/RmiUiTc6) API（OpenAI 兼容接口），使用免费模型Qwen/Qwen3-8B进行测试，如果返回较慢可自行更换其他模型。



In [5]:

API_KEY = "sk-vedrofottwzpxveidegfstxtmtjibwiknziaqenxyqexqkdx"
BASE_URL = "https://api.siliconflow.cn/v1"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# 快速测试
try:
    resp = client.chat.completions.create(
        model="Qwen/Qwen3-8B",
        messages=[{"role": "user", "content": "用一句话回答：什么是 NLP？"}],
        temperature=0.3,
        max_tokens=80
    )
    print("[OK] API 连通正常")
    print("模型回复：", resp.choices[0].message.content)
    print(f"Token 用量：{resp.usage.total_tokens}")
except Exception as e:
    print(f"[FAIL] API 调用失败：{e}")
    print("请检查 API_KEY 是否正确填写")

[OK] API 连通正常
模型回复： 

自然语言处理（NLP）是人工智能的一个分支，旨在使计算机能够理解、解析、生成和响应人类语言，通过算法和模型实现人机之间的高效沟通与信息处理。
Token 用量：246


---

## Part 1：语言模型演进与 Transformer 概览

**受众**：所有学生
**前置条件**：Part 0 完成
**学习目标**：理解语言模型的基本任务与演进路线，建立对 Transformer 的宏观认知

**核心问题**：AI 是怎么"理解"文字的？为什么 ChatGPT 能聊得这么自然？


### 1.1 语言模型的核心任务

语言模型的根本任务：**计算一个词序列（句子）出现的概率**，等价于"给定前文，预测下一个词"。

数学表达（概率链式法则）：

$$P(S) = P(w_1) \cdot P(w_2|w_1) \cdot P(w_3|w_1,w_2) \cdots P(w_m|w_1,...,w_{m-1})$$

**类比**：就是文字接龙游戏——给定前文，猜下一个最可能的词。


In [48]:
# 用 N-gram 直观感受"预测下一个词"
# 迷你语料库
from collections import defaultdict
corpus = [
    "buaa student learns",
    "buaa student works",
    "buaa student builds"
]


# 统计 Bigram（前一个词 → 下一个词 的频率）
bigram = defaultdict(lambda: defaultdict(int))
for sentence in corpus:
    words = sentence.split()
    for i in range(len(words) - 1):
        bigram[words[i]][words[i+1]] += 1

# 给定 "buaa student"，预测下一个词
prev_word = "agent"
print(f"给定前词 '{prev_word}'，下一个词的概率分布：")
total = sum(bigram[prev_word].values())
for word, count in sorted(bigram[prev_word].items(), key=lambda x: -x[1]):
    print(f"  {word}: {count}/{total} = {count/total:.3f}")

print()
print("这就是最原始的'语言模型'——基于统计计数预测下一个词。")
print("现代大模型用 Transformer 做同样的事，只是规模和精度高了几个数量级。")

给定前词 'agent'，下一个词的概率分布：
  learns: 1/3 = 0.333
  works: 1/3 = 0.333
  builds: 1/3 = 0.333

这就是最原始的'语言模型'——基于统计计数预测下一个词。
现代大模型用 Transformer 做同样的事，只是规模和精度高了几个数量级。


### 1.2 语言模型演进路线

| 阶段     | 代表模型        | 核心思想            | 缺陷                 |
| -------- | --------------- | ------------------- | -------------------- |
| 统计时代 | N-gram          | 计数 + 马尔可夫假设 | 无法处理未见组合     |
| 神经网络 | RNN/LSTM        | 隐藏状态 + 记忆机制 | 顺序计算慢、长程遗忘 |
| **当代** | **Transformer** | 自注意力 + 并行计算 | 计算量大、需海量数据 |

**Transformer 的核心突破**：自注意力机制让模型处理每个词时能"看到"整个句子，并且可以并行计算。

> **本讲只做概念性介绍**，Transformer 的架构细节（Q/K/V、多头注意力、位置编码等）将在第4讲深入展开。


### 1.3 Decoder-Only 架构：GPT 系列的选择

GPT 的核心思想：**只保留解码器，专注"预测下一个词"**。

- 回答问题、写故事、生成代码，本质上都是文字接龙
- 因果掩码（Causal Mask）：只能看到当前位置之前的词，不能偷看后面
- 自回归生成：一个词一个词地往后写

今天使用的 ChatGPT、DeepSeek、Qwen 全部基于 Decoder-Only 架构。


---

## Part 2：分词（Tokenization）与 BPE 算法

**受众**：所有学生
**前置条件**：Part 1 完成
**学习目标**：

1. 理解为什么需要分词，以及三种分词策略的取舍
2. 亲手实现 BPE 算法，理解子词分词的核心思想
3. 会用 tiktoken 计算 Token 数，管理上下文预算

**核心问题**：计算机只认识数字，"你好"对模型来说是什么？


### 2.1 为什么需要分词

**分词 = 将文本转换为模型可处理的数字序列**

三种策略对比：

| 方案               | 示例                   | 优点           | 缺点                   |
| ------------------ | ---------------------- | -------------- | ---------------------- |
| 按词分             | ["Hello", "world"]     | 直观           | 词表爆炸、无法处理新词 |
| 按字符分           | ["H","e","l","l","o"]  | 词表小         | 单字符无语义、序列太长 |
| **子词分（主流）** | ["Hello", "wor", "ld"] | 平衡词表与语义 | 需算法设计             |

现代大模型统一采用**子词分词**：常见词保留完整，罕见词拆成有意义的子词片段。


### 2.2 BPE 算法原理

**BPE（Byte-Pair Encoding）= 贪心合并最高频相邻对**

训练过程（迷你语料 `{"hug":1, "pug":1, "pun":1, "bun":1}`，目标词表大小 10）：

1. 初始化：每个词拆成字符，词表 = {h, u, g, p, n, b}（6个）
2. 统计相邻 token 对频率，合并最高频对
3. 重复步骤 2，直到达到目标词表大小

| 步骤 | 最高频对   | 合并后                   | 词表大小 |
| ---- | ---------- | ------------------------ | -------- |
| 1    | u+g → ug   | h ug, p ug, p u n, b u n | 7        |
| 2    | u+n → un   | h ug, p ug, p un, b un   | 8        |
| 3    | h+ug → hug | hug, p ug, p un, b un    | 9        |
| 4    | p+ug → pug | hug, pug, p un, b un     | 10       |

对未见过的词 "bug"：b 在词表，ug 在词表 → `['b', 'ug']`


### 2.3 BPE 算法 Python 实现


In [49]:
def train_bpe(corpus, num_merges):
    """
    BPE 训练：迭代合并最高频相邻 token 对

    参数：
        corpus: dict, {word: frequency}
        num_merges: int, 合并次数（决定最终词表大小）
    返回：
        merges: list, 合并历史（按顺序记录每次合并的 token 对）
        vocab: dict, {word: 分词后的 token 列表}
    """
    # 初始化：每个词拆成字符列表
    vocab = {word: list(word) for word in corpus}
    # 初始词表 = 所有出现过的字符
    base_vocab = set()
    for word in corpus:
        base_vocab.update(word)
    base_vocab = sorted(base_vocab)

    merges = []

    for step in range(num_merges):
        # 统计所有相邻 token 对的频率
        pairs = {}
        for word, freq in corpus.items():
            tokens = vocab[word]
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i+1])
                pairs[pair] = pairs.get(pair, 0) + freq

        if not pairs:
            break

        # 找到频率最高的 pair
        best_pair = max(pairs, key=pairs.get)
        new_token = best_pair[0] + best_pair[1]
        merges.append(best_pair)

        # 在所有词中执行合并
        for word in vocab:
            tokens = vocab[word]
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == best_pair:
                    new_tokens.append(new_token)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            vocab[word] = new_tokens

        print(f"步骤 {step+1}: 合并 {best_pair} → '{new_token}'  "
              f"(频率={pairs[best_pair]})")

    final_vocab = base_vocab + [m[0]+m[1] for m in merges]
    return merges, vocab, final_vocab


# 在迷你语料上训练
corpus = {"hug": 1, "pug": 1, "pun": 1, "bun": 1}
print("=== BPE 训练过程 ===")
print(f"语料：{corpus}")
print()
merges, vocab, final_vocab = train_bpe(corpus, num_merges=4)
print()
print(f"最终词表（大小 {len(final_vocab)}）：{final_vocab}")
print(f"分词结果：")
for word, tokens in vocab.items():
    print(f"  {word} → {tokens}")

=== BPE 训练过程 ===
语料：{'hug': 1, 'pug': 1, 'pun': 1, 'bun': 1}

步骤 1: 合并 ('u', 'g') → 'ug'  (频率=2)
步骤 2: 合并 ('u', 'n') → 'un'  (频率=2)
步骤 3: 合并 ('h', 'ug') → 'hug'  (频率=1)
步骤 4: 合并 ('p', 'ug') → 'pug'  (频率=1)

最终词表（大小 10）：['b', 'g', 'h', 'n', 'p', 'u', 'ug', 'un', 'hug', 'pug']
分词结果：
  hug → ['hug']
  pug → ['pug']
  pun → ['p', 'un']
  bun → ['b', 'un']


### 2.4 用训练好的 BPE 分词新词


In [50]:
def bpe_encode(word, merges):
    """用训练好的 BPE 规则对任意词进行分词"""
    tokens = list(word)
    for merge_pair in merges:
        new_token = merge_pair[0] + merge_pair[1]
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == merge_pair:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


# 对未见过的词进行分词
test_words = ["bug", "bugs", "hugs", "bun"]
print("=== 用 BPE 分词新词 ===")
for w in test_words:
    tokens = bpe_encode(w, merges)
    print(f"  {w} → {tokens}")

=== 用 BPE 分词新词 ===
  bug → ['b', 'ug']
  bugs → ['b', 'ug', 's']
  hugs → ['hug', 's']
  bun → ['b', 'un']


### 2.5 tiktoken 实操：Token 计算

`tiktoken` 是 OpenAI 开源的快速分词库，可以复现 GPT-3.5/GPT-4 的分词结果。


In [51]:

import tiktoken

# 加载 GPT-4 使用的分词器
enc = tiktoken.encoding_for_model("gpt-4")

# 中英文 Token 计算对比
texts = {
    "英文短句": "Hello, how are you?",
    "中文短句": "你好，最近怎么样？",
    "代码片段": "print('hello world')",
    "长文本": "自然语言处理是人工智能的重要分支，" * 5
}

print("=== Token 计算对比 ===")
print(f"{'类型':<10} {'字符数':<8} {'Token数':<8} {'比值(字符/Token)':<15}")
print("-" * 50)
for label, text in texts.items():
    n_chars = len(text)
    n_tokens = len(enc.encode(text))
    ratio = n_chars / n_tokens if n_tokens > 0 else 0
    print(f"{label:<10} {n_chars:<8} {n_tokens:<8} {ratio:<15.2f}")

print()
print("观察：中文的字符/Token 比通常低于英文，即中文更'费 Token'")

=== Token 计算对比 ===
类型         字符数      Token数   比值(字符/Token)   
--------------------------------------------------
英文短句       19       6        3.17           
中文短句       9        10       0.90           
代码片段       20       5        4.00           
长文本        85       85       1.00           

观察：中文的字符/Token 比通常低于英文，即中文更'费 Token'


In [52]:
# 查看 Token 的内部表示
text_zh = "你好，大模型"
tokens_zh = enc.encode(text_zh)

print(f"原文：{text_zh}")
print(f"Token IDs：{tokens_zh}")
print(f"Token 数：{len(tokens_zh)}")
print()
print("逐 Token 解码：")
for tid in tokens_zh:
    print(f"  ID={tid:>6}  →  '{enc.decode([tid])}'")

原文：你好，大模型
Token IDs：[57668, 53901, 3922, 27384, 54872, 25287]
Token 数：6

逐 Token 解码：
  ID= 57668  →  '你'
  ID= 53901  →  '好'
  ID=  3922  →  '，'
  ID= 27384  →  '大'
  ID= 54872  →  '模'
  ID= 25287  →  '型'


### 2.6 Token 对开发者的实际意义

| 维度            | 影响                   | 实际场景                              |
| --------------- | ---------------------- | ------------------------------------- |
| **上下文窗口**  | 模型能"看到"的最大范围 | GPT-4o: 128K tokens ≈ 10万字中文      |
| **计费**        | API 按 Token 计费      | DeepSeek-V3.2: ¥4/百万 input tokens  |
| **Prompt 长度** | 决定能给模型多少信息   | 长文档需截断或分块       |

**实用技巧**：

- 用 tiktoken 预估成本：`tokens × 单价`
- 中文 Prompt 比等长英文更"贵"（Token 数更多）
- 上下文预算分配：System Prompt 占 10-20%，用户输入 30-40%，预留生成空间

In [53]:
# 实战：估算一段 Prompt 的 API 成本
def estimate_cost(text, price_per_million=1.0):
    """估算一段文本的 Token 数和 API 成本"""
    n_tokens = len(enc.encode(text))
    cost = n_tokens / 1_000_000 * price_per_million
    return n_tokens, cost


# 示例：一段系统 Prompt
system_prompt = """你是一位资深的 Python 数据分析专家，拥有 10 年经验。
回答时使用中文，给出代码示例，并解释背后的设计考量。
只回答 Python 相关问题，对于其他问题请礼貌地说明你只擅长 Python。"""

n_tokens, cost = estimate_cost(system_prompt)
print(f"System Prompt 字符数：{len(system_prompt)}")
print(f"Token 数：{n_tokens}")
print(f"假设单价 ¥1/百万 Token，单次调用成本：¥{cost:.6f}")
print(f"调用 10000 次的总成本：¥{cost * 10000:.4f}")

System Prompt 字符数：100
Token 数：78
假设单价 ¥1/百万 Token，单次调用成本：¥0.000078
调用 10000 次的总成本：¥0.7800


---

## Part 3：采样参数（Temperature / Top-k / Top-p）

**受众**：所有学生
**前置条件**：Part 2 完成
**学习目标**：

1. 理解 Temperature、Top-k、Top-p 三个采样参数的原理
2. 通过对比实验直观感受参数对输出的影响
3. 学会根据任务类型选择合适的参数组合

**核心问题**：为什么同样的 Prompt，AI 有时给出不同答案？


### 3.1 采样参数的本质

大模型在每一步预测下一个 Token 时，输出的不是"哪个词"，而是"所有候选词的概率分布"。

采样参数控制**如何从这个分布中抽取下一个 Token**：

- **Temperature**：调整分布的"陡峭度"
- **Top-k**：只保留概率最高的 k 个候选
- **Top-p**：保留累积概率达到 p 的最小候选集合

**Softmax 公式**：$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$
**带温度的 Softmax**：$p_i^{(T)} = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}}$


### 3.2 Temperature 详解

| Temperature       | 分布形态 | 效果               | 适用场景                     |
| ----------------- | -------- | ------------------ | ---------------------------- |
| 0 ~ 0.3（低温）   | 更陡峭   | 确定性高，重复率高 | 代码生成、事实问答、数据提取 |
| 0.3 ~ 0.7（中温） | 适中     | 自然平衡           | 日常对话、邮件撰写           |
| 0.7 ~ 2.0（高温） | 更平坦   | 创意发散           | 诗歌、头脑风暴               |

- **T=0**：始终选概率最高的 Token，输出完全确定（适合代码/抽取）
- **T=1**：标准 Softmax，原汁原味
- **T>1**：低概率词被放大，输出更多样但可能不连贯


### 3.3 Top-k 与 Top-p 详解

**Top-k**：只从概率最高的 k 个词中采样

- k=1：等同于贪心解码，完全确定
- k=50：从前 50 个候选词中按概率采样

**Top-p（核采样）**：累积概率达到 p 就停止

- p=0.9：保留累积概率达到 90% 的最小词集合
- 动态调整候选数量（概率集中时候选少，分散时候选多）

**组合使用优先级**：Temperature → Top-k → Top-p

实践中常同时使用 `temperature=0.7, top_p=0.9`，Top-k 默认不启用或设为 -1。


### 3.4 采样参数对比实验


In [8]:
def chat(prompt, temperature=0.7, top_p=0.9, max_tokens=100, n=1):
    """调用大模型，返回 n 个候选回复"""
    resp = client.chat.completions.create(
        model="Qwen/Qwen3-8B",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        n=n
    )
    return [c.message.content for c in resp.choices]

In [ ]:
# 实验1：Temperature 对比
prompt = "用一句话描述春天"

print("=== Temperature 对比实验 ===")
print(f"Prompt: {prompt}")
print()

for temp in [0.1, 0.7, 1.5]:
    print(f"--- Temperature = {temp} ---")
    try:
        replies = chat(prompt, temperature=temp, n=3, max_tokens=60)
        for i, r in enumerate(replies, 1):
            print(f"  候选{i}: {r}")
    except Exception as e:
        print(f"  调用失败: {e}")
    print()

=== Temperature 对比实验 ===
Prompt: 用一句话描述春天

--- Temperature = 0.1 ---
  候选1: 

春天是大地披上彩衣，万物在暖阳中苏醒，生机在枝头绽放的诗意时节。
  候选2: 

春风轻抚大地，万物复苏绽放，生机盎然如诗如画。
  候选3: 

春风轻拂，万物复苏，繁花似锦，绿意盎然，大地披上新装，生机勃勃。

--- Temperature = 0.7 ---
  候选1: 

春天是大地脱下冬装、万物复苏的时节，枝头绽放的花朵与泥土中萌发的嫩芽共同谱写着生命的序章。
  候选2: 

春天是万物复苏的季节，大地披上绿装，花儿竞相绽放，空气中弥漫着生机与希望。
  候选3: 

春风轻拂，万物复苏，繁花似锦，生机盎然。

--- Temperature = 1.5 ---
  候选1: 

春风轻抚，万物苏醒，繁花点缀青山，暖意悄然浸润人间。
  候选2: 

春风拂面，万物复苏，花香四溢中孕育着无限生机。
  候选3: 

春天是万物悄然苏醒、繁花次第绽放的生机盎然时节。



In [9]:
# 实验2：同一 Prompt 多次调用，观察 Temperature=0 的稳定性
prompt = "是啊吃什么"

print("=== Temperature=0 稳定性验证 ===")
print(f"Prompt: {prompt}")
print()
print("三次调用结果（通常应更一致；若仍有差异，说明服务端并非完全确定性）：")
print()

results = []
for i in range(3):
    try:
        r = chat(prompt, temperature=0, top_p=1, max_tokens=200)[0].strip()
        results.append(r)
        print(f"--- 第 {i+1} 次 ---")
        print(r[:200] + ("..." if len(r) > 200 else ""))
        print()
    except Exception as e:
        print(f"第 {i+1} 次调用失败: {e}")

# 验证是否完全相同
if len(results) >= 2:
    all_equal = all(r == results[0] for r in results[1:])
    print(f"三次结果是否完全相同：{all_equal}")
    print()
    if all_equal:
        print("结论：在当前模型与服务端条件下，Temperature=0 结果稳定。")
    else:
        print("结论：Temperature=0 只能降低随机性，不能绝对保证每次都完全一致。")
        print("如果业务需要强一致性，建议：固定模型版本、关闭流式、设置 seed（若接口支持）、并对输出做规范化与校验。")

=== Temperature=0 稳定性验证 ===
Prompt: 是啊吃什么

三次调用结果（通常应更一致；若仍有差异，说明服务端并非完全确定性）：

--- 第 1 次 ---
你今天想吃什么类型的食物呢？比如健康餐、快餐、甜点，或者有特别的饮食需求？可以告诉我更多细节，我会根据你的喜好和情况推荐哦！ 

--- 第 2 次 ---
你好呀！ 你是在问今天想吃什么吗？还是有特别的场合或需求呢？比如：

- 想吃什么类型的菜？（中餐、西餐、甜点、轻食等）
- 有没有饮食限制？（比如素食、忌口、减肥等）
- 是想自己做还是点外卖/吃快餐？
- 有没有预算范围？

告诉我更多细节，我可以帮你推荐哦！

--- 第 3 次 ---
你今天想吃什么类型的食物呢？比如健康餐、快餐、甜点，或者有特别的饮食需求？可以告诉我更多细节，我会根据你的喜好和情况推荐哦！ 

三次结果是否完全相同：False

结论：Temperature=0 只能降低随机性，不能绝对保证每次都完全一致。
如果业务需要强一致性，建议：固定模型版本、关闭流式、设置 seed（若接口支持）、并对输出做规范化与校验。


### 3.5 采样参数选择指南

| 任务类型 | Temperature | Top-p | 说明         |
| -------- | ----------- | ----- | ------------ |
| 代码生成 | 0 ~ 0.2     | 0.9   | 需要确定性   |
| 事实问答 | 0 ~ 0.3     | 0.9   | 避免胡编     |
| 信息抽取 | 0           | 0.9   | 严格按格式   |
| 日常对话 | 0.5 ~ 0.7   | 0.9   | 自然流畅     |
| 创意写作 | 0.8 ~ 1.2   | 0.95  | 鼓励发散     |
| 头脑风暴 | 1.0 ~ 1.5   | 0.95  | 最大化多样性 |

**工具线衔接**：用 Inline Edit（Cmd+K）快速切换参数值，对比输出差异，建立对参数的直觉。


---

## Part 4：Prompt Engineering 核心技巧

**受众**：所有学生
**前置条件**：Part 3 完成
**学习目标**：

1. 掌握 Zero-shot / One-shot / Few-shot 的差异与适用场景
2. 掌握角色扮演（System Prompt）的设计方法
3. 掌握思维链（CoT）的使用时机
4. 掌握结构化输出（JSON）的约束技巧
5. 能根据任务选择合适的 Prompt 策略

**核心问题**：怎么让 AI 稳定地按我想要的格式输出？


### 4.1 Prompt Engineering 的定位

**Prompt = 与大模型沟通的"编程语言"**

| 传统编程            | Prompt Engineering   |
| ------------------- | -------------------- |
| 写 Python/Java 代码 | 写自然语言指令       |
| 编译器执行          | 大模型执行           |
| 语法错误会报错      | 表达不清得到错误输出 |
| 确定性结果          | 概率性结果（需约束） |

**与第1讲两段式 Prompt 的关系**：

- 第1讲：先氛围后约束（一种简单的迭代策略）
- 本讲：升级为完整的方法论体系（Zero/Few-shot、CoT、结构化输出等）


### 4.2 Zero-shot / One-shot / Few-shot

根据给模型提供的示例数量分类：

| 类型      | 示例数 | 适用场景             |
| --------- | ------ | -------------------- |
| Zero-shot | 0      | 简单任务、模型能力强 |
| One-shot  | 1      | 需要格式示范         |
| Few-shot  | 2-5    | 任务边界复杂         |

示例越多，模型对任务边界理解越准确，但 Token 消耗也越大。


In [10]:
# 任务：情感分类（正面/负面/中性）

# ===== Zero-shot =====
zero_shot_prompt = """判断以下文本的情感倾向，只输出"正面"、"负面"或"中性"。

文本：北航计算机学院的 Python 进阶课程内容很扎实！"""

print("=== Zero-shot ===")
print(f"Prompt:\n{zero_shot_prompt}")
print()
try:
    r = chat(zero_shot_prompt, temperature=0, max_tokens=20)[0]
    print(f"输出: {r}")
except Exception as e:
    print(f"调用失败: {e}")
print()

# ===== Few-shot =====
few_shot_prompt = """判断文本的情感倾向。

文本：这家餐厅的服务太慢了。
情感：负面

文本：今天天气不错，心情很好。
情感：正面

文本：快递已送达。
情感：中性

文本：北航计算机学院的 Python 进阶课程内容很扎实！
情感："""

print("=== Few-shot ===")
print(f"Prompt（节选）:\n{few_shot_prompt[:150]}...")
print()
try:
    r = chat(few_shot_prompt, temperature=0, max_tokens=20)[0]
    print(f"输出: {r}")
except Exception as e:
    print(f"调用失败: {e}")

=== Zero-shot ===
Prompt:
判断以下文本的情感倾向，只输出"正面"、"负面"或"中性"。

文本：北航计算机学院的 Python 进阶课程内容很扎实！

输出: 

正面

=== Few-shot ===
Prompt（节选）:
判断文本的情感倾向。

文本：这家餐厅的服务太慢了。
情感：负面

文本：今天天气不错，心情很好。
情感：正面

文本：快递已送达。
情感：中性

文本：北航计算机学院的 Python 进阶课程内容很扎实！
情感：...

输出: 

情感：正面

解析：文本中使用了“很扎实”这一明确的褒义


### 4.3 角色扮演（System Prompt）

通过赋予模型角色，控制回答风格、知识范围和语气。

**System Prompt 设计三要素**：

| 要素     | 示例                     | 作用         |
| -------- | ------------------------ | ------------ |
| 身份定义 | "你是资深 Python 专家"   | 激活相关知识 |
| 行为约束 | "回答用中文，附代码"     | 控制输出格式 |
| 边界限制 | "只回答 Python 相关问题" | 防止跑题     |

**工具线衔接**：System Prompt 就是第1讲 Rules 文件的"API 版本"——从文件控制升级为消息控制。


In [57]:
# 对比有无 System Prompt 的输出差异

question = "如何高效处理百万行 CSV？"

# 无 System Prompt
print("=== 无 System Prompt ===")
try:
    r = chat(question, temperature=0.3, max_tokens=200)[0]
    print(r)
except Exception as e:
    print(f"调用失败: {e}")
print()

# 有 System Prompt
system_prompt = """你是一位资深 Python 数据分析专家，拥有 10 年经验。
回答时使用中文，给出可运行的代码示例，并解释背后的设计考量。
回答简洁，不超过 200 字。"""

print("=== 有 System Prompt ===")
try:
    resp = client.chat.completions.create(
        model="Qwen/Qwen3-8B",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        temperature=0.3,
        max_tokens=400
    )
    print(resp.choices[0].message.content)
except Exception as e:
    print(f"调用失败: {e}")

=== 无 System Prompt ===


处理百万行CSV文件时，需要考虑性能、内存管理和处理效率。以下是高效处理的分步指南和方法：

---

### **1. 选择合适的工具**
根据需求选择工具，避免一次性加载全部数据到内存：
- **Python**（推荐）：使用 `pandas`、`csv` 模块或 `Dask`。
- **命令行工具**：如 `awk`、`csvkit`（`csvsql`、`csvcut`）。
- **数据库**：将CSV导入数据库（如SQLite、PostgreSQL）后查询。
- **大数据工具**：如 `Hadoop`、`Spark`（适用于超大规模数据）。

---

### **2. 分块处理（Chunking）**
避免一次性加载整个文件到内存，分块处理可显著降低内存压力：
#### **Python + Pandas**
```python
import pandas as pd

chunksize = 100000

=== 有 System Prompt ===


处理百万行CSV可采用分块读取+惰性计算。推荐使用Dask或pandas结合chunksize：

```python
import pandas as pd
chunksize = 100000
for chunk in pd.read_csv('large_file.csv', chunksize=chunksize):
    # 处理每个chunk
    chunk['new_col'] = chunk['col1'] + chunk['col2']
    # 累积结果或直接写入数据库
```

设计考量：1. 避免一次性加载全部数据到内存 2. 使用数值类型优化（如指定dtype） 3. 仅加载所需列（usecols） 4. 通过并行计算（Dask）提升处理速度。


### 4.4 思维链（Chain-of-Thought, CoT）

**核心思想**：引导模型"一步一步地思考"，提升推理准确率。

**实现方式**：在 Prompt 中加一句引导语，如"请一步一步地思考"。

**CoT 适用场景决策**：

| 任务类型      | 推荐策略                       |
| ------------- | ------------------------------ |
| 简单分类/提取 | Zero-shot 或 Few-shot          |
| 需要推理/计算 | CoT                            |
| 格式严格      | 结构化输出                     |
| 复杂多步骤    | CoT + Few-shot + System Prompt |


In [58]:
# 推理任务对比：直接回答 vs CoT

problem = """一个篮球队在一个赛季的80场比赛中赢了60%。
在接下来的赛季中，他们打了15场比赛，赢了12场。
两个赛季的总胜率是多少？"""

# 直接回答
print("=== 直接回答（无 CoT）===")
try:
    r = chat(problem + "\n请直接给出答案。", temperature=0, max_tokens=100)[0]
    print(r)
except Exception as e:
    print(f"调用失败: {e}")
print()

# CoT
print("=== 思维链（CoT）===")
try:
    r = chat(problem + "\n请一步一步地思考并解答。", temperature=0, max_tokens=300)[0]
    print(r)
except Exception as e:
    print(f"调用失败: {e}")

=== 直接回答（无 CoT）===


两个赛季的总胜率是63.16%。  

**答案：63.16%**

=== 思维链（CoT）===


一个篮球队在一个赛季的80场比赛中赢了60%，即：

$$
80 \times 60\% = 80 \times 0.6 = 48 \text{场胜利}
$$

在接下来的赛季中，他们打了15场比赛，赢了12场，即：

$$
12 \text{场胜利}
$$

两个赛季的总胜利场数为：

$$
48 + 12 = 60 \text{场}
$$

总比赛场数为：

$$
80 + 15 = 95 \text{场}
$$

因此，两个赛季的总胜率为：

$$
\frac{60}{95} = \frac{12}{19} \approx 0.6316
$$

将小数转换为百分比并四舍五入到两位小数：

$$
0.6316 \times 100\% \approx 63.16\%
$$

**答案：两个赛季的总胜率约为63.16%。**


### 4.5 结构化输出

让模型按指定格式返回结果，便于程序解析（`json.loads`）。

**关键技巧**：

1. 明确给出目标格式（JSON / XML / 表格）
2. 提供一个完整示例
3. 加约束："严格按照上述格式，不要添加额外内容"
4. 设置 `temperature=0` 提高格式稳定性


In [59]:
# 结构化输出：从产品评论中提取信息

review = '这款"星尘"笔记本电脑屏幕显示效果惊人，但我不太喜欢它的键盘手感，续航也只能算一般。'

system_prompt = """你是信息抽取专家。从产品评论中提取以下字段，严格按 JSON 格式输出，不要添加任何其他内容。

输出格式：
{
  "product_name": "产品名称",
  "sentiment": "正面/负面/混合",
  "pros": ["优点1", "优点2"],
  "cons": ["缺点1"],
  "rating": 1-5的整数
}"""

print("=== 结构化输出 ===")
print(f"评论: {review}")
print()
try:
    resp = client.chat.completions.create(
        model="Qwen/Qwen3-8B",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": review}
        ],
        temperature=0,
        max_tokens=200
    )
    raw_output = resp.choices[0].message.content
    print(f"模型原始输出:\n{raw_output}")
    print()

    # 尝试解析 JSON
    try:
        data = json.loads(raw_output)
        print("[OK] JSON 解析成功：")
        print(json.dumps(data, ensure_ascii=False, indent=2))
    except json.JSONDecodeError as e:
        print(f"[FAIL] JSON 解析失败: {e}")
        print("（实际使用时可加 retry 或用正则提取）")
except Exception as e:
    print(f"调用失败: {e}")

=== 结构化输出 ===
评论: 这款"星尘"笔记本电脑屏幕显示效果惊人，但我不太喜欢它的键盘手感，续航也只能算一般。

模型原始输出:


{
  "product_name": "星尘笔记本电脑",
  "sentiment": "混合",
  "pros": ["屏幕显示效果惊人"],
  "cons": ["键盘手感不佳", "续航能力一般"],
  "rating": 3
}

[OK] JSON 解析成功：
{
  "product_name": "星尘笔记本电脑",
  "sentiment": "混合",
  "pros": [
    "屏幕显示效果惊人"
  ],
  "cons": [
    "键盘手感不佳",
    "续航能力一般"
  ],
  "rating": 3
}


### 4.6 Prompt Engineering 策略决策树

```text
你的任务是什么？
├── 简单分类/判断 → Zero-shot（直接问）
├── 需要特定输出风格 → Few-shot（给示例）
├── 需要推理/计算 → CoT（"请逐步思考"）
├── 需要稳定格式 → 结构化输出（给 JSON 模板）
├── 复杂多步骤任务 → CoT + Few-shot + System Prompt
└── 不确定 → 先试 Zero-shot，效果不好再升级
```

**迭代原则**：从最简单的策略开始，根据效果逐步加码。

**与 Vibe Coding 的映射**：

| Vibe Coding 技巧 | PE 方法论            | 本质         |
| ---------------- | -------------------- | ------------ |
| 两段式 Prompt    | Zero-shot → Few-shot | 迭代细化     |
| Rules 文件       | System Prompt        | 全局风格控制 |
| @文件引用        | Few-shot 上下文注入  | 精准信息供给 |
| Diff 审查        | 幻觉检测             | 输出质量把关 |


---

## Part 5：大模型 API 调用实战

**受众**：所有学生
**前置条件**：Part 4 完成
**学习目标**：

1. 掌握 OpenAI 兼容 API 的 messages 结构
2. 能封装函数完成情感分类、信息抽取、文本摘要三类 NLP 任务
3. 学会异常处理与成本统计

**核心问题**：如何把 Prompt Engineering 技巧落地为可复用的代码？


### 5.1 消息结构

```python
messages = [
    {"role": "system", "content": "..."},     # 系统指令（对应 Rules）
    {"role": "user", "content": "..."},       # 用户输入（对应 Prompt）
    {"role": "assistant", "content": "..."},  # AI 历史回复（多轮对话）
    {"role": "user", "content": "..."}        # 下一轮输入
]
```

**三种角色**：

- `system`：全局指令，定义 AI 的身份和行为（对应第1讲 Rules 文件）
- `user`：用户的每一轮输入（对应 Chat 框中的 Prompt）
- `assistant`：AI 的历史回复（多轮对话上下文）


### 5.2 任务一：情感分类


In [60]:
def classify_sentiment(text):
    """使用 Few-shot + 结构化输出完成情感分类

    参数：
        text: 待分类的文本
    返回：
        dict: {"sentiment": "正面/负面/中性", "confidence": 0.0-1.0}
    """
    system_prompt = """你是情感分析专家。对输入文本判断情感，严格按 JSON 格式输出。
输出格式：{"sentiment": "正面/负面/中性", "confidence": 0.0到1.0的浮点数}
不要添加任何其他内容。"""

    user_prompt = f"""文本："{text}"
输出："""

    try:
        resp = client.chat.completions.create(
            model="Qwen/Qwen3-8B",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0,
            max_tokens=80
        )
        raw = resp.choices[0].message.content.strip()
        return json.loads(raw), resp.usage.total_tokens
    except json.JSONDecodeError:
        return {"error": "JSON 解析失败", "raw": raw}, 0
    except Exception as e:
        return {"error": str(e)}, 0


# 测试
test_reviews = [
    "这个产品真的太好用了，强烈推荐！",
    "质量很差，用了一周就坏了，客服态度也差。",
    "今天收到了快递，包装完好。",
    "性价比一般，谈不上好也谈不上差。"
]

print("=== 情感分类测试 ===")
total_tokens = 0
for text in test_reviews:
    result, tokens = classify_sentiment(text)
    total_tokens += tokens
    print(f"文本: {text}")
    print(f"结果: {result}  (Token: {tokens})")
    print()

print(f"总 Token 消耗: {total_tokens}")

=== 情感分类测试 ===
文本: 这个产品真的太好用了，强烈推荐！
结果: {'sentiment': '正面', 'confidence': 0.95}  (Token: 392)

文本: 质量很差，用了一周就坏了，客服态度也差。
结果: {'sentiment': '负面', 'confidence': 0.9}  (Token: 472)

文本: 今天收到了快递，包装完好。
结果: {'sentiment': '正面', 'confidence': 0.8}  (Token: 508)

文本: 性价比一般，谈不上好也谈不上差。
结果: {'sentiment': '中性', 'confidence': 0.9}  (Token: 499)

总 Token 消耗: 1871


### 5.3 任务二：信息抽取


In [61]:
def extract_info(review):
    """从产品评论中提取结构化信息

    参数：
        review: 产品评论文本
    返回：
        dict: 包含 product/pros/cons/rating 字段
    """
    system_prompt = """你是信息抽取专家。从产品评论中提取以下字段，严格按 JSON 格式输出：
{
  "product": "产品名称",
  "pros": ["优点列表"],
  "cons": ["缺点列表"],
  "rating": 1-5的整数
}
不要添加任何其他内容。"""

    try:
        resp = client.chat.completions.create(
            model="Qwen/Qwen3-8B",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": review}
            ],
            temperature=0,
            max_tokens=200
        )
        raw = resp.choices[0].message.content.strip()
        return json.loads(raw), resp.usage.total_tokens
    except Exception as e:
        return {"error": str(e), "raw": raw if 'raw' in dir() else ''}, 0


# 测试
reviews = [
    "这款华为 MatePad 平板续航超强，能用一整天，但是扬声器音质一般，看视频体验打折扣。",
    "小米手环8 戴着很舒服，睡眠监测准确，就是屏幕亮度在阳光下不够。"
]

print("=== 信息抽取测试 ===")
for r in reviews:
    result, tokens = extract_info(r)[0], extract_info(r)[1]
    print(f"评论: {r[:50]}...")
    print(f"结果: {json.dumps(result, ensure_ascii=False, indent=2)}")
    print(f"Token: {tokens}")
    print()

=== 信息抽取测试 ===
评论: 这款华为 MatePad 平板续航超强，能用一整天，但是扬声器音质一般，看视频体验打折扣。...
结果: {
  "product": "华为 MatePad 平板",
  "pros": [
    "续航超强",
    "能用一整天"
  ],
  "cons": [
    "扬声器音质一般",
    "看视频体验打折扣"
  ],
  "rating": 4
}
Token: 795

评论: 小米手环8 戴着很舒服，睡眠监测准确，就是屏幕亮度在阳光下不够。...
结果: {
  "product": "小米手环8",
  "pros": [
    "戴着很舒服",
    "睡眠监测准确"
  ],
  "cons": [
    "屏幕亮度在阳光下不够"
  ],
  "rating": 4
}
Token: 1150



### 5.4 任务三：文本摘要（CoT 策略）


In [2]:
def summarize(text, max_words=50):
    """使用 CoT 策略生成摘要

    参数：
        text: 待摘要的长文本
        max_words: 摘要最大字数
    返回：
        str: 摘要文本
    """
    system_prompt = f"""你是摘要专家。请按以下步骤生成摘要：
1. 识别文本的核心主题
2. 提取 3 个关键信息点
3. 用不超过 {max_words} 字组织成流畅的摘要
只输出最终摘要，不需要展示中间步骤。"""

    try:
        resp = client.chat.completions.create(
            model="Qwen/Qwen3-8B",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": text}
            ],
            temperature=0.3,
            max_tokens=200
        )
        return resp.choices[0].message.content.strip(), resp.usage.total_tokens
    except Exception as e:
        return f"调用失败: {e}", 0


# 测试
news = """近日，北京航空航天大学计算机学院推出全新 AI 编程进阶课程《Python 进阶》。
该课程系统覆盖 Pandas 数据分析、机器学习、NLP 基础与 Prompt 工程，采用 Vibe Coding 理念，
借助 CloudStudio + CodeBuddy 平台让学生通过自然语言驱动 AI 完成真实数据分析项目。
"""

print("=== 文本摘要测试 ===")
print(f"原文（{len(news)} 字）:\n{news}")
print()
summary, tokens = summarize(news, max_words=50)
print(f"摘要:\n{summary}")
print(f"\nToken 消耗: {tokens}")

=== 文本摘要测试 ===
原文（161 字）:
近日，北京航空航天大学计算机学院推出全新 AI 编程进阶课程《Python 进阶》。
该课程系统覆盖 Pandas 数据分析、机器学习、NLP 基础与 Prompt 工程，采用 Vibe Coding 理念，
借助 CloudStudio + CodeBuddy 平台让学生通过自然语言驱动 AI 完成真实数据分析项目。


摘要:
调用失败: name 'client' is not defined

Token 消耗: 0


### 5.5 三类任务的 PE 策略对比

| 任务     | Temperature | 策略                        | 关键技巧     |
| -------- | ----------- | --------------------------- | ------------ |
| 情感分类 | 0           | Few-shot + JSON             | 给示例锁格式 |
| 信息抽取 | 0           | System Prompt + JSON Schema | 明确字段定义 |
| 文本摘要 | 0.3         | CoT + 长度约束              | 步骤引导     |

**工程化要点**：

- 统一封装函数，入参清晰、返回 JSON
- 异常处理：API 失败、JSON 解析失败都要兜底
- 成本统计：用 `response.usage.total_tokens` 跟踪消耗


---

## Part 6：模型选择与局限性

**受众**：所有学生
**前置条件**：Part 5 完成
**学习目标**：

1. 掌握模型选型的四大维度
2. 了解主流闭源/开源模型的特点
3. 理解缩放法则与能力涌现
4. 理解模型幻觉的成因与缓解策略


### 6.1 模型选型四维度

| 维度           | 关键问题           | 权衡点                   |
| -------------- | ------------------ | ------------------------ |
| **性能**       | 任务能力够不够？   | 推理/代码/中文等专项能力 |
| **成本**       | Token 单价多少？   | 输入/输出分别计费        |
| **速度**       | 响应延迟可接受吗？ | 首 Token 延迟、生成速度  |
| **上下文长度** | 需要处理多长文档？ | 4K / 32K / 128K / 1M     |

**选型决策树**：

```text
需要最强推理/代码能力？ → 是 → GPT-5.5 / Claude Opus 4.8 / DeepSeek-V4-Pro
→ 否 ↓
数据敏感或需本地部署？ → 是 → Qwen3 / LLaMA 4 / DeepSeek-V4-Flash 开源版（第5讲）
→ 否 ↓
预算有限？ → 是 → DeepSeek-V4-Flash / Qwen3-8B（性价比高）
→ 否 → 闭源 API（开箱即用）
```

### 6.2 主流闭源模型概览（2026 年）

| 模型               | 厂商        | 特点                      | 上下文  |
| ---------------- | --------- | ----------------------- | ---- |
| GPT-5.5          | OpenAI    | 最新旗舰，推理强，1M 上下文         | 1M   |
| Claude Opus 4.8  | Anthropic | 长文档强，安全性高               | 200K |
| Gemini 2.5 Pro   | Google    | 原生多模态，超长上下文             | 2M   |
| 通义千问 Qwen3.5     | 阿里        | 混合推理，中文优秀，成本低           | 256K |
| 智谱 GLM-5.2       | 智谱        | 744B MoE，Agent 能力强，MIT 开源 | 200K |
| DeepSeek-V4-Flash | 深度求索     | 284B MoE，轻量快速，性价比高       | 1M   |
| DeepSeek-V4-Pro  | 深度求索      | 1.6T MoE 旗舰，推理能力顶尖      | 1M   |

### 6.3 主流开源模型概览（2026 年）

| 模型                  | 参数规模         | 特点                              | 适用场景          |
| --------------------- | ------------ | -------------------------------- | ------------- |
| LLaMA 4               | 17B-405B     | Meta 出品，原生多模态，生态成熟              | 研究、定制化        |
| Qwen3（通义千问）      | 0.6B-235B    | 阿里出品，混合推理，中文强，多尺寸              | 中文应用、边缘到云端部署  |
| DeepSeek-V4-Flash     | 284B (MoE)   | MIT 开源，轻量快速，1M 上下文，性价比极高       | 日常任务、高并发、成本敏感 |
| DeepSeek-V4-Pro       | 1600B (MoE)  | MIT 开源，旗舰推理，1M 上下文              | 复杂推理、数学、Agent |
| GLM-5.2（智谱）        | 744B (MoE)   | MIT 开源，Agent 能力强，200K 上下文       | 中文应用、代码、逻辑推理  |
| Mistral / Mixtral     | 7B-141B(MoE) | 欧洲出品，小尺寸高性能，多语言                | 资源受限环境        |

**开源 vs 闭源**：

- 闭源：开箱即用、性能前沿、按量计费、数据需上传
- 开源：可本地部署、可微调、数据安全、性能略弱

**课程使用的模型（硅基流动 API）**：

| 模型 ID                        | 定价                   | 说明                       |
| ------------------------------ | ---------------------- | -------------------------- |
| `Qwen/Qwen3-8B`                | ≈ ¥0.3/百万 tokens     | 课程主力，轻量快速，中文理解好 |
| `THUDM/GLM-Z1-9B-0414`         | **免费**               | 备用，零成本适合大量调试      |
| `deepseek-ai/DeepSeek-V3.2`    | ¥4/百万 input tokens   | 高精度任务可选，性能更强      |

> 开源模型的本地部署与微调将在**第5讲**深入展开。

### 6.4 缩放法则（Scaling Laws）

**核心发现**：模型性能与参数量(N)、数据量(D)、计算量(C) 呈幂律关系

$$L(N) \propto N^{-\alpha}, \quad L(D) \propto D^{-\beta}, \quad L(C) \propto C^{-\gamma}$$

**三个关键洞察**：

1. **持续投入有回报**：按比例增加参数/数据/算力，性能可预测提升
2. **Chinchilla 定律**（DeepMind 2022）：给定算力预算下，参数量和数据量存在最优配比——70B 模型 + 4倍数据，性能超越 175B 的 GPT-3
3. **能力涌现**：模型规模达阈值后突然出现新能力（CoT 推理、指令遵循、代码生成）

**对开发者的启示**：

- 选模型不能只看参数量，还要看训练数据量
- 复杂任务需要足够大的模型才能"涌现"出能力


### 6.5 模型幻觉（Hallucination）

**幻觉 = 模型自信地生成与事实矛盾或不存在的内容**

**三种典型幻觉**：

| 类型       | 示例                             | 特点               |
| ---------- | -------------------------------- | ------------------ |
| 事实性幻觉 | 虚构论文引用、不存在的事件       | 最常见，最危险     |
| 忠实性幻觉 | 总结时偏离原文、添加未提及内容   | RAG 系统高发       |
| 逻辑幻觉   | 推理链中间步骤出错但结论看似合理 | CoT 也无法完全避免 |

**产生原因**：

- 训练数据含错误/矛盾信息
- 自回归生成本质是"预测下一个最可能的词"，无事实核查
- 知识时效性：训练截止后的事件模型不知道

**四层缓解策略**：

| 层级      | 策略                  | 示例                            |
| --------- | --------------------- | ------------------------------- |
| Prompt 层 | 明确边界              | "如果不确定，请说'我不知道'"    |
| 推理层    | CoT 自我验证          | 分步推理后再检查每一步          |
| 系统层    | **RAG 检索增强**      | 先查文档再回答 |
| 工程层    | 多模型投票 + 人工审核 | 关键场景必须人工把关            |


---

## Part 7：知识点速查表与工具速查表

### 知识点速查表

| 知识点 | 关键内容 |
| --- | --- |
| 语言模型 | 计算词序列概率，预测下一个词 |
| Transformer | 自注意力 + 并行计算，Decoder-Only 是主流 |
| 分词 | 文本 → 数字序列，子词分词是主流 |
| BPE | 贪心合并最高频相邻 token 对 |
| Token | 模型处理的最小单元，影响上下文/计费/长度 |
| Temperature | 控制随机性，0=确定，>1=发散 |
| Top-k | 只从概率最高的 k 个词中采样 |
| Top-p | 累积概率达到 p 就停止（核采样） |
| Zero-shot | 不给示例直接下指令 |
| Few-shot | 给多个示例锁格式 |
| CoT | "请一步一步地思考"，提升推理 |
| 结构化输出 | 给 JSON 模板，便于程序解析 |
| 缩放法则 | 性能与参数/数据/算力呈幂律 |
| 模型幻觉 | 自信编造不存在的事实 |

### Prompt Engineering 策略速查表

| 策略 | 适用场景 | 关键写法 |
| --- | --- | --- |
| Zero-shot | 简单任务、模型能力强 | 直接下指令 |
| One-shot | 需要格式示范 | 给一个示例 |
| Few-shot | 任务边界复杂 | 给 3-5 个示例 |
| 角色扮演 | 需要专业风格 | "你是一位资深..." |
| CoT | 推理/计算任务 | "请一步一步地思考" |
| 结构化输出 | 需程序解析 | 给 JSON 模板 |
| 约束边界 | 防幻觉 | "仅基于以下文档..." |

### 采样参数速查表

| 参数 | 范围 | 作用 | 推荐值 |
| --- | --- | --- | --- |
| Temperature | 0-2 | 控制随机性 | 代码:0 / 对话:0.7 / 创意:1.0 |
| Top-k | 1-100 | 候选词数量 | 40 (通用) |
| Top-p | 0-1 | 累积概率阈值 | 0.9 (通用) |
| max_tokens | - | 最大生成长度 | 根据任务设 |

### OpenAI 兼容 API 速查表

| 操作 | 代码 |
| --- | --- |
| 创建客户端 | `client = OpenAI(api_key=..., base_url=...)` |
| 对话补全 | `client.chat.completions.create(model=..., messages=...)` |
| 取回复内容 | `response.choices[0].message.content` |
| 取 Token 用量 | `response.usage.total_tokens` |
| tiktoken 编码 | `enc = tiktoken.encoding_for_model("gpt-4")` |
| 计算 Token 数 | `len(enc.encode(text))` |

---

## 本讲总结

**知识线收获**：

- 语言模型从 N-gram → RNN/LSTM → Transformer 的演进
- 分词原理（BPE 算法）与 tiktoken 工具
- 采样参数（Temperature / Top-k / Top-p）的原理与调参
- Prompt Engineering 完整方法论（Zero/Few-shot、CoT、结构化输出）
- OpenAI 兼容 API 调用与三类 NLP 任务实战
- 模型选择四维度与缩放法则、幻觉缓解

